Import required modules and libraries

In [ ]:
#Import the required modules and libraries
import pastaq as pq
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as colors
from collections import defaultdict
from pathlib import Path
import re
from itertools import combinations
import os
import ms_entropy as me
from pathlib import Path
from sklearn.metrics import root_mean_squared_error
import ast

In [ ]:
import faulthandler
faulthandler.enable() 

In [ ]:
pip install py-spy

In [ ]:
pip install debugpy

In [ ]:
import debugpy

Import helpers for PASTAQ. Here the helper functions have been added to a .py file called "PASTAQ_helpers" to keep the notebook tidy. 

In [ ]:
import PASTAQ_helpers

Define the input path to the raw .ms2 files and the output path.

In [ ]:
input_files = [{'raw_path': r"C:\Users\diego.DESKTOP-7OSFK5B\Documents\[MSc Research Project 2]\Code\pastaq_TEST\raw\b1.ms2"},
                {'raw_path': r"C:\Users\diego.DESKTOP-7OSFK5B\Documents\[MSc Research Project 2]\Code\pastaq_TEST\raw\p1_1.ms2"}
               ]

In [ ]:
output_dir = r"C:\Users\diego.DESKTOP-7OSFK5B\Documents\[MSc Research Project 2]\Code\pastaq_TEST"

Define the path to the "feature_clusters_annotations.csv" generated by PASTAQ's DDA pipeline

In [ ]:
feature_clusters_annotations_csv = pd.read_csv(r"C:\Users\diego.DESKTOP-7OSFK5B\Documents\[MSc Research Project 2]\Code\pastaq_TEST\quant\feature_clusters_annotations.csv")

Link the MS2 information to the csv file containing the feature clusters, this function has been updated to allow the user to decide if they wish to keep the features with no linked MS2. If you wish to remove the features that have no linked MS2, please set "MS2_only" to "True". If you wish to keep all the features (including those with no linked MS2), please set "MS2_only" to "False". It will default to "False" if you do not change it.

In [ ]:
def combine_multiple_samples(
    feature_clusters_annotations_csv,
    input_files,
    output_dir,
    MS2_only=False
):

    annotations_lookup = defaultdict(list)
    no_ms2_features = []

    # ---------------------------------------------------------------------------
    # Build lookup table (to speed up program)
    # ---------------------------------------------------------------------------
    for row in feature_clusters_annotations_csv.itertuples(index=False):

        if pd.notna(row.msms_id):

            key = (
                row.file_id,
                row.msms_id
            )

            annotations_lookup[key].append(row)

        else:

            # Separate the features with no linked MS2
            if not MS2_only:
                no_ms2_features.append(row)


    combined_multiple_samples = []

    # ---------------------------------------------------------------------------
    # Add the features that have no linked MS2 (if MS2 only has not been set to "True")
    # ---------------------------------------------------------------------------
    if not MS2_only:

        for row in no_ms2_features:

            combined_multiple_samples.append({
                "cluster_id": row.cluster_id,
                "file_id": row.file_id,
                "feature_id": row.feature_id,
                "peak_id": row.peak_id,
                "msms_id": None,
                "ms2_rt": None,
                "charge_state": row.charge_state,
                "ms2_peaks": None,
                "flags": ["No linked MS2"]
            })


    # ------------------------------------------
    # Process raw files for MS2
    # ------------------------------------------
    for file in input_files:

        if "stem" not in file:

            file["stem"] = os.path.splitext(
                os.path.basename(
                    file["raw_path"]
                )
            )[0]

        stem = file["stem"]

        in_path = os.path.join(
            output_dir,
            "raw",
            f"{stem}.ms2"
        )

        if not os.path.exists(in_path):
            continue


        raw_data = pq.read_raw_data(in_path)


        # Only process scans that are needed
        needed_scans = {
            key[1]
            for key in annotations_lookup
            if key[0] == stem
        }


        for scan in raw_data.scans:

            if scan.scan_number not in needed_scans:
                continue


            key = (
                stem,
                scan.scan_number
            )

            annotations = annotations_lookup.get(key)

            if not annotations:
                continue


            flags = []

            ms2_mz = scan.mz
            ms2_intensity = scan.intensity
            ms2_rt = scan.retention_time


            # ------------------------------------------
            # Check MS2 validity
            # ------------------------------------------
            invalid_ms2 = (
                ms2_mz is None
                or ms2_intensity is None
                or len(ms2_mz) == 0
                or len(ms2_intensity) == 0
                or len(ms2_mz) != len(ms2_intensity)
            )


            if invalid_ms2:

                flags.append(
                    "No valid linked MS2"
                )

                if MS2_only:
                    continue

                ms2_peaks = None


            else:

                mz_array = np.asarray(
                    ms2_mz,
                    dtype=np.float32
                )

                intensity_array = np.asarray(
                    ms2_intensity,
                    dtype=np.float32
                )

                sorted_idx = np.argsort(
                    mz_array
                )

                ms2_peaks = np.column_stack(
                    (
                        mz_array[sorted_idx],
                        intensity_array[sorted_idx]
                    )
                ).astype(np.float32)


            # ------------------------------------------
            # Add MS2 details to annotations
            # ------------------------------------------
            append_result = combined_multiple_samples.append

            for row in annotations:

                append_result({
                    "cluster_id": row.cluster_id,
                    "file_id": row.file_id,
                    "feature_id": row.feature_id,
                    "peak_id": row.peak_id,
                    "msms_id": row.msms_id,
                    "ms2_rt": ms2_rt,
                    "charge_state": row.charge_state,
                    "ms2_peaks": ms2_peaks,
                    "flags": flags.copy()
                })


    return combined_multiple_samples

In [ ]:
combined_multiple_samples = combine_multiple_samples(feature_clusters_annotations_csv, input_files, output_dir)

Specify the input pathways for the ".features" files created by PASTAQs' DDA pipeline.

In [ ]:
input_files = [{'raw_path': r"C:\Users\diego.DESKTOP-7OSFK5B\Documents\[MSc Research Project 2]\Code\pastaq_TEST\features\b1.features"},
                {'raw_path': r"C:\Users\diego.DESKTOP-7OSFK5B\Documents\[MSc Research Project 2]\Code\pastaq_TEST\features\p1_1.features"}
                ]

In [ ]:
output_dir = r"C:\Users\diego.DESKTOP-7OSFK5B\Documents\[MSc Research Project 2]\Code\pastaq_TEST"

Link the MS1 features to the large dataframe now containing both the MS2 information and the feature cluster information.

In [ ]:
# Function to link the MS1 features to the large dataframe now containing both the MS2 information and the feature cluster informations:
def link_features(combined_multiple_samples, input_files, output_dir):
    # Preprocess: build a lookup dictionary to avoid filtering the DataFrame each time
    annotations_lookup = defaultdict(list)
    linked_features = []

    for item in combined_multiple_samples:
        if pd.notnull(item['feature_id']):
            key = (item['file_id'], item['feature_id'])
            annotations_lookup[key].append(item)

    for file in input_files:
        if 'stem' not in file:
            base_name = os.path.splitext(os.path.basename(file['raw_path']))[0]
            file['stem'] = base_name
        stem = file['stem']
        in_path_features = os.path.join(output_dir, 'features', f"{stem}.features")

        if not os.path.exists(in_path_features):
            print('missing feature file/s')
            continue

        features = pq.read_features(in_path_features)

        for feature in features:
            id = feature.id
            key = (stem, id)
            annotations = annotations_lookup.get(key)

            if not annotations:
                continue

            if isinstance(annotations, list):
                for annotation in annotations:
                    linked_features.append({
                        'cluster_id': annotation['cluster_id'],
                        'file_id': annotation['file_id'],
                        'feature_id': id,
                        'feature_peak_ids': feature.peak_ids,
                        'peak_id' : annotation['peak_id'],
                        'msms_id': annotation['msms_id'],
                        'precursor_mz' : feature.monoisotopic_mz,
                        'precursor_rt': feature.monoisotopic_rt,
                        'precursor_intensity': feature.monoisotopic_height,
                        'precursor_vol': feature.monoisotopic_volume,
                        'total_intensity': feature.total_height,
                        'total_volume': feature.total_volume,
                        'average_ms1_mz': feature.average_mz,
                        'average_rt': feature.average_rt,
                        "ms2_rt": annotation["ms2_rt"],
                        'charge_state': feature.charge_state,
                        'flags' : annotation["flags"],
                        'ms2_sample_peaks': annotation['ms2_peaks'],                  
                        })

    return linked_features

In [ ]:
linked_features = link_features(combined_multiple_samples=combined_multiple_samples, input_files=input_files, output_dir=output_dir)

Convert "linked_features" to a PANDAS dataframe

In [ ]:
linked_features_df = pd.DataFrame(linked_features)
print(linked_features_df)

Load your file containing your internal standards. It should be in the format in the example excel file supplied.

In [ ]:
#Load file with list of internal standards
IS_file = r"C:\Users\diego.DESKTOP-7OSFK5B\OneDrive\Belgeler\Internal_Standards_File-Pos.xlsx"
IS_data = pd.read_excel(IS_file)

Load your MSP file containing the information needed for lipid identification. This workflow was tested using the MSP file from massbank. 

In [ ]:
#Load msp file - mass bank file
# msp_file = r"C:\Users\diego.DESKTOP-7OSFK5B\Documents\[MSc Research Project 2]\MassBankMSP\MsMs_Positive_LabOnly_AllDirectories.msp"

In [ ]:
#Load msp file
msp_file = r"C:\Users\diego.DESKTOP-7OSFK5B\Documents\[MSc Research Project 2]\Code\MSPs\MSDIAL-TandemMassSpectralAtlas-VS69-Pos.msp"

Run the functions to read your MSP file.

In [ ]:
# Function to parse any strings that may be present in the MSP file
def parse_array_from_string(s):
    if isinstance(s, str):
        return np.array([float(x) for x in re.findall(r"[-+]?\d*\.\d+|\d+", s)])
    return np.array([])

#V2 -> added class to function
# Function to read and retrieve the annotations from an MSP file in the .msp format 
def read_msp_file(file_path):
    with open(file_path, 'r') as file:
        lines = file.readlines()
    
    spectra_data = []
    current_spectrum = {}
    peak_data_started = False
    
    for line in lines:
        line = line.strip()

        if line.startswith("Num Peaks"):
            peak_data_started = True
            continue

        if not peak_data_started:
            if line.startswith("NAME:"):
                text = line.split(":", 1)[1].strip()   # Remove "NAME:"

                if "RIKEN P-VS1" in text:
                    current_spectrum["class"] = "Unknown"
                else:
                # Remove "low score:" from class if it is present
                    text = re.sub(r"^low score:\s*", "", text, flags=re.IGNORECASE)
                    text = re.sub(r"^no ms2:\s*", "", text, flags=re.IGNORECASE)

                # Extract the first word (the lipid class)
                    match = re.match(r"[A-Za-z0-9-]+", text)
                    current_spectrum["class"] = match.group(0) if match else "unknown"

                if "|" in line:
                    # If the line contains '|', split the line after the '|' character
                    parts = line.split('|')
                    current_spectrum['name'] = parts[0].replace("NAME:", "").strip()  # Remove "NAME:" and strip any extra spaces
                    current_spectrum['saturation'] = parts[1].strip()  # After the '|'
                    
                else:
                    # Otherwise, just use the name if no saturation is specified
                    current_spectrum['name'] = line.split(":", 1)[1].strip()

               
            elif line.startswith("PRECURSORMZ:"):
                current_spectrum['precursor_mz'] = float(line.split(":", 1)[1].strip())
            elif line.startswith("PRECURSORTYPE:"):
                current_spectrum['precursor_type'] = line.split(":", 1)[1].strip()
            elif line.startswith("IONMODE:"):
                current_spectrum['ion_mode'] = line.split(":", 1)[1].strip()              
            elif line.startswith("RETENTIONTIME:"):
                _, raw = line.split(":", 1)
                val = raw.strip()
                if not val:
                    print("[SKIP] empty retention_time")
                    continue
                try:
                    rt = float(val)
                except ValueError:
                        rt_parsed = parse_array_from_string(val)
                        if isinstance(rt_parsed, (list, np.ndarray)) and len(rt_parsed) == 1:
                            rt = float(rt_parsed[0])
                        else:
                            print("[SKIP] array invalid, skipping")
                            continue
                current_spectrum['retention_time'] = rt * 60 # convert to seconds - REMOVE this line if your retention times are already in seconds
            elif line.startswith("Name: "):
                if current_spectrum:
                    spectra_data.append(current_spectrum)
                    current_spectrum = {"Name": line.split(":",1)[1].strip()}
            elif line.startswith("FORMULA:"):
                current_spectrum['formula'] = line.split(":", 1)[1].strip()
            elif line.startswith("INCHIKEY:"):
                current_spectrum['inchi_key'] = line.split(":", 1)[1].strip()
            elif line.startswith("SMILES:"):
                current_spectrum['smiles'] = line.split(":", 1)[1].strip()
            elif line.startswith("COMMENT:"):
                current_spectrum['comment'] = line.split(":", 1)[1].strip()

        else:
            try:
                mz, intensity = map(float, line.split())
                current_spectrum.setdefault('peaks', []).append((mz, intensity))
            except ValueError:
                # This is where the spectrum data is stored and new spectrum begins
                if current_spectrum:
                    spectra_data.append(current_spectrum)
                current_spectrum = {}  # Reset for the next spectrum
                peak_data_started = False  # Reset peak reading flag

    # Add last spectrum if it exists
    if current_spectrum:
        spectra_data.append(current_spectrum)
    
    return spectra_data

Read your MSP file.

In [ ]:
msp_data = read_msp_file(msp_file)

In [ ]:
print(msp_data)

Preprocess and sort the MSP library

In [ ]:
#V2
#Preprocess the msp library to speed up the program
#This will remove any annotations with missing m/z and/or retention times
def preprocess_msp_library(msp_data):

    valid_annotations = []
    msp_mz = []

    for annotation in msp_data:

        precursor_mz = annotation.get("precursor_mz")
        retention_time = annotation.get("retention_time")
        peaks = annotation.get("peaks")

        if precursor_mz is None or retention_time is None or peaks is None:
            continue

        peaks = np.asarray(peaks, dtype=np.float32)

        if peaks.ndim == 3 and peaks.shape[0] == 1:
            peaks = peaks[0]

        if peaks.ndim != 2:
            continue

        annotation["_mz"] = precursor_mz
        annotation["_rt"] = retention_time
        annotation["_peaks_array"] = peaks

        valid_annotations.append(annotation)
        msp_mz.append(precursor_mz)

    msp_mz = np.asarray(msp_mz, dtype=np.float32)
    sorted_idx = np.argsort(msp_mz)

    return (
        valid_annotations,
        msp_mz[sorted_idx],
        sorted_idx
    )

In [ ]:
#V3
def preprocess_msp_library(msp_data):

    valid_annotations = []

    mz_values = []
    rt_values = []
    peaks_arrays = []

    for annotation in msp_data:

        precursor_mz = annotation.get("precursor_mz")
        retention_time = annotation.get("retention_time")
        peaks = annotation.get("peaks")

        if (
            precursor_mz is None
            or retention_time is None
            or peaks is None
        ):
            continue

        peaks = np.asarray(peaks, dtype=np.float32)

        if peaks.ndim == 3 and peaks.shape[0] == 1:
            peaks = peaks[0]

        if peaks.ndim != 2:
            continue

        valid_annotations.append(annotation)

        mz_values.append(precursor_mz)
        rt_values.append(retention_time)
        peaks_arrays.append(peaks)

    mz_values = np.asarray(mz_values, dtype=np.float32)
    rt_values = np.asarray(rt_values, dtype=np.float32)

    sorted_idx = np.argsort(mz_values)

    return (
        valid_annotations,
        mz_values,
        rt_values,
        peaks_arrays,
        mz_values[sorted_idx],
        sorted_idx
    )

In [ ]:
preprocessed_msp_library = preprocess_msp_library(msp_data)

DEBUGGING (runs for V2, needs to be changed to test V3)

In [ ]:
valid_annotations, mz_values, rt_values, peaks_arrays, mz_values, sorted_idx = preprocessed_msp_library

print(len(valid_annotations))
print(msp_mz_sorted.shape)
print(sorted_idx.shape)

In [ ]:
print(valid_annotations[30000])

Detect your internal standards. This function assumes that the retention time of your internal standards is given in minutes. NB: Please remove the "*60" part of the "IS_rt" code if your retention time is already in seconds. 

In [ ]:
#Function to detect internal standards from excel file
def detect_internal_standard(
    feature,
    IS_records,
    IS_mz_tolerance=0.005,
    IS_rt_tolerance=10.0
):

    precursor_mz = feature["precursor_mz"]
    precursor_rt = feature["precursor_rt"]

    best_match = None
    best_score = np.inf

    for record in IS_records:
        IS_mz = record["IS_mz"]
        IS_rt = record["IS_avg_RT"] * 60

        if pd.isna(IS_mz) or pd.isna(IS_rt):
            continue

        IS_mz_distance = abs(
            precursor_mz - IS_mz
        )

        IS_rt_distance = abs(
            precursor_rt - IS_rt
        )

        if (
            IS_mz_distance > IS_mz_tolerance
            or IS_rt_distance > IS_rt_tolerance
        ):
            continue

        IS_match_score = (IS_mz_distance / IS_mz_tolerance + IS_rt_distance / IS_rt_tolerance)

        if IS_match_score < best_score:

            best_score = IS_match_score

            best_match = {
                "IS_No": record["IS_No"],
                "IS_Name": record["IS_Name"],
                "IS_Class": record["IS_Class"],
                "IS_mz": IS_mz,
                "IS_RT": IS_rt,
                "IS_match_score": float(IS_match_score),
                "IS_mass_error_ppm":
                    ((IS_mz - precursor_mz) / IS_mz)
                    * 1e6
            }

    return best_match

In [ ]:
output_dir = r"C:\Users\diego.DESKTOP-7OSFK5B\Documents\[MSc Research Project 2]\Code\pastaq_TEST"

In [ ]:
#V1
def find_best_msp_match(
    feature,
    valid_annotations,
    msp_mz_sorted,
    sorted_idx,
    mz_tolerance,
    rt_tolerance
):
    cluster_id = feature['cluster_id']
    file_id = feature['file_id']
    feature_id = feature['feature_id']
    feature_peak_ids = feature['feature_peak_ids']
    peak_id = feature['peak_id']
    msms_id = feature['msms_id']
    precursor_mz = feature['precursor_mz']
    precursor_rt = feature['precursor_rt']
    precursor_intensity = feature['precursor_intensity']
    precursor_volume = feature['precursor_vol']
    average_ms1_mz = feature['average_ms1_mz']
    average_rt = feature['average_rt']
    charge_state = feature['charge_state']
    raw_ms2_sample_peaks = feature['ms2_sample_peaks']
    total_intensity = feature['total_intensity']
    total_volume = feature['total_volume']

    has_ms2 = raw_ms2_sample_peaks is not None

    # ----------------------------------------------
    # Prepare MS2 data if available
    # ----------------------------------------------
    if has_ms2:

        centroided_peaks = me.clean_spectrum(
            raw_ms2_sample_peaks,
            min_ms2_difference_in_da=0.02,
            normalize_intensity=False
        )

        centroided_arr = np.array(centroided_peaks)
        centroided_arr_list = centroided_arr.tolist()

        peaks_query = np.asarray(
            raw_ms2_sample_peaks,
            dtype=np.float32
        )

        sample_mz, sample_intensity = zip(
            *centroided_arr_list
        )

        sample_mz = np.asarray(sample_mz)

        sample_intensity = np.asarray(
            sample_intensity,
            dtype=np.float32
        )

    else:

        centroided_arr_list = []
        peaks_query = None

    # ----------------------------------------------
    # m/z candidate lookup from MSP
    # ----------------------------------------------
    left = np.searchsorted(
        msp_mz_sorted,
        precursor_mz - mz_tolerance,
        side="left"
    )

    right = np.searchsorted(
        msp_mz_sorted,
        precursor_mz + mz_tolerance,
        side="right"
    )

    candidate_indices = sorted_idx[left:right]

    best_annotation = None
    best_score = np.inf
    best_mass_error_ppm = None
    best_rmse_mz = None

    # ----------------------------------------------
    # Pass 1:
    # Find best candidate using only mz/rt score
    # ----------------------------------------------
    for idx in candidate_indices:

        annotation = valid_annotations[idx]

        rt_distance = abs(
            precursor_rt - annotation["_rt"]
        )

        if rt_distance > rt_tolerance:
            continue

        mz_distance = abs(
            precursor_mz - annotation["_mz"]
        )

        match_score = (mz_distance / mz_tolerance + rt_distance / rt_tolerance)

        if match_score < best_score:

            best_score = match_score
            best_annotation = annotation

            best_mass_error_ppm = (
                (annotation["_mz"] - precursor_mz)
                / annotation["_mz"]
            ) * 1e6

            best_rmse_mz = mz_distance

    # ----------------------------------------------
    # If no match is found
    # ----------------------------------------------
    if best_annotation is None:
        return None

    # ----------------------------------------------
    # Calculate spectral similarity for features that have linked MS2 peaks
    # ----------------------------------------------
    if has_ms2:

        peaks_reference = best_annotation["_peaks_array"]

        if peaks_reference is None:
            raise ValueError(
                f"Missing _peaks_array for annotation "
                f"{best_annotation.get('name')}"
            )

        unweighted_similarity = (
            me.calculate_unweighted_entropy_similarity(
                peaks_query,
                peaks_reference
            )
        )

        similarity = (
            me.calculate_entropy_similarity(
                peaks_query,
                peaks_reference
            )
        )

        # ------------------------------------------
        # Dot product similarity
        # ------------------------------------------
        reference_mz = peaks_reference[:, 0]
        reference_intensity = peaks_reference[:, 1]

        matched_indices = np.searchsorted(
            reference_mz,
            sample_mz
        )

        matched_indices = matched_indices[
            matched_indices < len(reference_mz)
        ]

        sample_spectrum = sample_intensity[
            :len(matched_indices)
        ]

        reference_spectrum = reference_intensity[
            matched_indices
        ]

        if (
            len(sample_spectrum) < 3
            or len(reference_spectrum) < 3
        ):
            dot_product = None

        else:

            sample_norm = np.linalg.norm(
                sample_spectrum
            )

            reference_norm = np.linalg.norm(
                reference_spectrum
            )

            if (
                sample_norm == 0
                or reference_norm == 0
            ):
                dot_product = None

            else:

                dot_product = float(
                    np.dot(
                        sample_spectrum
                        / sample_norm,
                        reference_spectrum
                        / reference_norm
                    )
                )

    else:

        unweighted_similarity = "No MS2"
        similarity = "No MS2"
        dot_product = "No MS2"

    # ----------------------------------------------
    # Match record
    # ----------------------------------------------
    best_match = {
        "score": float(best_score),
        "class": best_annotation.get("class"),
        "name": best_annotation.get("name"),
        "saturation": best_annotation.get(
            "saturation"
        ),
        "retention_time": best_annotation.get(
            "retention_time"
        ),
        "precursor_mz": best_annotation.get(
            "precursor_mz"
        ),
        "precursor_type": best_annotation.get(
            "precursor_type"
        ),
        "smiles": best_annotation.get(
            "smiles"
        ),
        "msp_peaks": best_annotation.get(
            "peaks"
        )
    }

    return {
        "file_id": file_id,
        "cluster_id": cluster_id,
        "feature_id": feature_id,
        "feature_peak_ids": feature_peak_ids,
        "peak_id": peak_id,
        "msms_id": msms_id,
        "precursor_mz": precursor_mz,
        "precursor_intensity":
            precursor_intensity,
        "precursor_rt": precursor_rt,
        "precursor_volume":
            precursor_volume,
        "charge_state": charge_state,
        "has_ms2": has_ms2,
        "mass_error_ppm":
            float(best_mass_error_ppm),
        "rmse_mz":
            float(best_rmse_mz),
        "cent_ms2_peaks":
            centroided_arr_list,
        "unweighted_entropy_similarity":
            unweighted_similarity,
        "entropy_similarity":
            similarity,
        "dot_product":
            dot_product
            if dot_product is not None
            else "NA",
        "match": best_match
    }

In [ ]:
#V2
# This probably does not run yet (it has not been tested yet)
#To do: update to also match based on MS2
def find_best_msp_match(
    feature,
    valid_annotations,
    mz_values,
    rt_values,
    peaks_arrays,
    msp_mz_sorted,
    sorted_idx,
    mz_tolerance,
    rt_tolerance
):
    cluster_id = feature['cluster_id']
    file_id = feature['file_id']
    feature_id = feature['feature_id']
    feature_peak_ids = feature['feature_peak_ids']
    peak_id = feature['peak_id']
    msms_id = feature['msms_id']
    precursor_mz = feature['precursor_mz']
    precursor_rt = feature['precursor_rt']
    precursor_intensity = feature['precursor_intensity']
    precursor_volume = feature['precursor_vol']
    average_ms1_mz = feature['average_ms1_mz'] #Do we want this?
    average_rt = feature['average_rt'] #Do we want this?
    charge_state = feature['charge_state']
    raw_ms2_sample_peaks = feature['ms2_sample_peaks']
    total_intensity = feature['total_intensity'] #Do we want this?
    total_volume = feature['total_volume'] #Do we want this?

    has_ms2 = raw_ms2_sample_peaks is not None

    # ----------------------------------------------
    # Prepare MS2 data if available
    # ----------------------------------------------
    if has_ms2:
        centroided_arr = np.asarray(
                me.clean_spectrum(raw_ms2_sample_peaks,
                            min_ms2_difference_in_da=0.02,
                            normalize_intensity=False),
                dtype=np.float32
                )

        sample_mz = centroided_arr[:,0]
        sample_intensity = centroided_arr[:,1]

        centroided_arr_list = centroided_arr.tolist()

    else:
        #What do we want here? To remove them?
        centroided_arr_list = []
        peaks_query = None

    # ----------------------------------------------
    # m/z candidate lookup from MSP
    # ----------------------------------------------
    left = np.searchsorted(
        msp_mz_sorted,
        precursor_mz - mz_tolerance,
        side="left"
    )

    right = np.searchsorted(
        msp_mz_sorted,
        precursor_mz + mz_tolerance,
        side="right"
    )

    candidate_indices = sorted_idx[left:right]

    candidate_mz = mz_values[candidate_indices]
    candidate_rt = rt_values[candidate_indices]

    rt_distance = np.abs(candidate_rt - precursor_rt)
    mz_distance = np.abs(candidate_mz - precursor_mz)

    mask = rt_distance <= rt_tolerance

    if not np.any(mask):
        return None

    scores = (
        mz_distance[mask] / mz_tolerance +
        rt_distance[mask] / rt_tolerance
    )

    best_local = np.argmin(scores)
    best_idx = candidate_indices[mask][best_local]

    best_annotation = None
    best_score = np.inf
    best_mass_error_ppm = None
    best_rmse_mz = None

    #To Do: Remove candidates that have no linked MS2 peaks
    #To Do: Add code to match based on MS2 similarity score + m/z and RT score
        #If ms2: use m/z and RT score + MS2 similarity score to find best match
        #IF only MS1: use m/z and RT score to find best match
    for idx in candidate_indices:

        annotation = valid_annotations[idx]

        peaks_reference = peaks_arrays[idx]

        rt_distance = abs(
            precursor_rt - rt_values[idx]
        )

        if rt_distance > rt_tolerance:
            continue

        mz_distance = abs(
            precursor_mz - mz_values[idx]
        )

        ms2_similarity_score = me.calculate_entropy_similarity(
                        peaks_query,
                        peaks_reference
                    )
                )

        match_score = (mz_distance / mz_tolerance + rt_distance / rt_tolerance + ms2_similarity_score)  # Placeholder for MS2 similarity score
        # Do we also want to weight MS2 similarity score?

        best_idx = None
        best_score = np.inf

        if match_score < best_score:

            best_score = match_score
            best_idx = idx

            best_mass_error_ppm = (
                (annotation["_mz"] - precursor_mz)
                / annotation["_mz"]
            ) * 1e6

            best_rmse_mz = mz_distance

    # ----------------------------------------------
    # If no match is found
    # ----------------------------------------------
    if best_idx is None:
        return None

    best_annotation = valid_annotations[best_idx]
    peaks_reference = peaks_arrays[best_idx]

    # ----------------------------------------------
    # Calculate spectral similarity for features that have linked MS2 peaks
    # To do: adapt code to use this also for identification
    # How do we want to do this?
    # First, match based on m/z and RT, then calculate the similarity score for the best match?
    # Or use m/z, RT and similarity score to find the best match?
    # ----------------------------------------------
    if has_ms2:

        peaks_reference = peaks_arrays[idx]

        if peaks_reference is None:
            raise ValueError(
                f"Missing _peaks_array for annotation "
                f"{best_annotation.get('name')}"
            )

        unweighted_similarity = (
            me.calculate_unweighted_entropy_similarity(
                peaks_query,
                peaks_reference
            )
        )

        similarity = (
            me.calculate_entropy_similarity(
                peaks_query,
                peaks_reference
            )
        )

        # ------------------------------------------
        # Dot product similarity
        # ------------------------------------------
        reference_mz = peaks_reference[:, 0]
        reference_intensity = peaks_reference[:, 1]

        matched_indices = np.searchsorted(
            reference_mz,
            sample_mz
        )

        matched_indices = matched_indices[
            matched_indices < len(reference_mz)
        ]

        sample_spectrum = sample_intensity[
            :len(matched_indices)
        ]

        reference_spectrum = reference_intensity[
            matched_indices
        ]

        if (
            len(sample_spectrum) < 3
            or len(reference_spectrum) < 3
        ):
            dot_product = None

        else:

            sample_norm = np.linalg.norm(
                sample_spectrum
            )

            reference_norm = np.linalg.norm(
                reference_spectrum
            )

            if (
                sample_norm == 0
                or reference_norm == 0
            ):
                dot_product = None

            else:

                dot_product = float(
                    np.dot(
                        sample_spectrum
                        / sample_norm,
                        reference_spectrum
                        / reference_norm
                    )
                )

    else:

        unweighted_similarity = "No MS2"
        similarity = "No MS2"
        dot_product = "No MS2"

    # ----------------------------------------------
    # Match record
    # ----------------------------------------------
    best_match = {
        "score": float(best_score),
        "class": best_annotation.get("class"),
        "name": best_annotation.get("name"),
        "saturation": best_annotation.get(
            "saturation"
        ),
        "retention_time": best_annotation.get(
            "retention_time"
        ),
        "precursor_mz": best_annotation.get(
            "precursor_mz"
        ),
        "precursor_type": best_annotation.get(
            "precursor_type"
        ),
        "smiles": best_annotation.get(
            "smiles"
        ),
        "msp_peaks": best_annotation.get(
            "peaks"
        )
    }

    return {
        "file_id": file_id,
        "cluster_id": cluster_id,
        "feature_id": feature_id,
        "feature_peak_ids": feature_peak_ids,
        "peak_id": peak_id,
        "msms_id": msms_id,
        "precursor_mz": precursor_mz,
        "precursor_intensity":
            precursor_intensity,
        "precursor_rt": precursor_rt,
        "precursor_volume":
            precursor_volume,
        "charge_state": charge_state,
        "has_ms2": has_ms2,
        "mass_error_ppm":
            float(best_mass_error_ppm),
        "rmse_mz":
            float(best_rmse_mz),
        "cent_ms2_peaks":
            centroided_arr_list,
        "unweighted_entropy_similarity":
            unweighted_similarity,
        "entropy_similarity":
            similarity,
        "dot_product":
            dot_product
            if dot_product is not None
            else "NA",
        "match": best_match
    }

Identify matches (internal standard or lipid/ metabolite) for features.

In [ ]:
def identify_matches(
    results,
    msp_data,
    IS_data,
    mz_tolerance=0.025,
    rt_tolerance=8.0,
    IS_mz_tolerance=0.01,
    IS_rt_tolerance=10.0,
    Unknowns=True
):

    identified_matches = []

    IS_records = IS_data.to_dict("records")

    (
        valid_annotations,
        msp_mz_sorted,
        sorted_idx
    ) = preprocess_msp_library(msp_data)

    for feature in results:
        file_id = feature["file_id"]
        cluster_id = feature["cluster_id"]
        feature_id = feature["feature_id"]
        peak_id = feature["peak_id"]
        precursor_mz = feature["precursor_mz"]
        precursor_rt = feature["precursor_rt"]
        charge_state = feature["charge_state"]
        precursor_intensity = feature["precursor_intensity"]
        precursor_volume = feature["precursor_vol"]
        raw_ms2_sample_peaks = feature["ms2_sample_peaks"]

        # -----------------------------
        # Internal Standard Search
        # -----------------------------
        is_match = detect_internal_standard(
            feature,
            IS_records,
            IS_mz_tolerance,
            IS_rt_tolerance
        )

        if is_match is not None:

            identified_matches.append({
                "file_id": file_id,
                "cluster_id": cluster_id,
                "feature_id": feature_id,
                "peak_id": peak_id,
                "precursor_mz": precursor_mz,
                "precursor_rt": precursor_rt,
                "charge_state": charge_state,
                "precursor_intensity": precursor_intensity,
                "precursor_volume": precursor_volume,
                "raw_ms2_sample_peaks": raw_ms2_sample_peaks,

                "classification": "internal_standard",

                "IS_match": is_match,
                "MSP_match": None,
                "Class": is_match["IS_Class"],
            })

            # Prevent MSP annotation
            continue

        # -----------------------------
        # MSP Search
        # -----------------------------
        msp_match = find_best_msp_match(
            feature,
            valid_annotations,
            msp_mz_sorted,
            sorted_idx,
            mz_tolerance,
            rt_tolerance
        )

        if msp_match is not None:

            identified_matches.append({
                "file_id": file_id,
                "cluster_id": cluster_id,
                "feature_id": feature_id,
                "peak_id": peak_id,
                "precursor_mz": precursor_mz,
                "precursor_rt": precursor_rt,
                "charge_state": charge_state,
                "precursor_intensity": precursor_intensity,
                "precursor_volume": precursor_volume,
                "raw_ms2_sample_peaks": raw_ms2_sample_peaks,

                "classification": "library_match",

                "IS_match": None,
                "MSP_match": msp_match,
                "Class": msp_match["match"]["class"],
            })

        elif Unknowns:

            identified_matches.append({
                "file_id": file_id,
                "cluster_id": cluster_id,
                "feature_id": feature_id,
                "peak_id": peak_id,
                "precursor_mz": precursor_mz,
                "precursor_rt": precursor_rt,
                "charge_state": charge_state,
                "precursor_intensity": precursor_intensity,
                "precursor_volume": precursor_volume,
                "raw_ms2_sample_peaks": raw_ms2_sample_peaks,

                "classification": "unknown",

                "IS_match": None,
                "MSP_match": None,
                "Class": "Unknown",
            })

        # If Unknowns=False and no MSP match,
        # feature is skipped automatically

    return identified_matches

In [ ]:
matches = identify_matches(
    linked_features,
    msp_data,
    IS_data,
    mz_tolerance=0.025,
    rt_tolerance=8.0,
    IS_mz_tolerance=0.005,
    IS_rt_tolerance=10.0,
    Unknowns=False
)

In [ ]:
print(matches)

Normalize_to_IS()
1) extract_group()
2) get_candidate_IS()
3) choose_best_IS()
4) calculate_normalized_values()

In [ ]:
import re
import numpy as np
from collections import defaultdict

#V4
def extract_group(file_id):
    return re.sub(r'[_-]?\d+$', '', file_id)


def IS_distance(feature, is_feature, mz_tolerance, rt_tolerance):

    mz_distance = abs(
        feature["precursor_mz"]
        - is_feature.get("precursor_mz")
    )

    rt_distance = abs(
        feature["precursor_rt"]
        - is_feature.get("precursor_rt")
    )

    return (mz_distance / mz_tolerance + rt_distance / rt_tolerance)


def calculate_cv(values):

    values = np.asarray(values, dtype=float)

    mean = np.mean(values)

    if mean == 0:
        return np.inf

    return np.std(values, ddof=1) / mean * 100


def choose_lowest_cv_IS(
    sample_group,
    candidate_ISs
):

    if len(candidate_ISs) == 1:
        return candidate_ISs[0]

    if not sample_group:
        return candidate_ISs[0]

    best_IS = None
    best_cv = np.inf

    for IS in candidate_ISs:

        IS_intensity = IS.get("precursor_intensity")

        if IS_intensity is None or IS_intensity == 0:
            continue

        normalized = [
            lipid["precursor_intensity"] / IS_intensity
            for lipid in sample_group
            if lipid.get("precursor_intensity") is not None
        ]

        if len(normalized) < 2:
            continue

        cv = calculate_cv(normalized)

        if cv < best_cv:
            best_cv = cv
            best_IS = IS

    return best_IS

def choose_best_IS(
    feature,
    candidates,
    rt_tolerance=30,
    mz_tolerance= 0.025,
    sample_group=None,
    normalize_to_group=True,
    normalize_to_class=False
):

    if not candidates:
        return None

    if len(candidates) == 1:
        return candidates[0]

    scores = sorted(
        [
            (IS_distance(feature, IS), IS)
            for IS in candidates
        ],
        key=lambda x: x[0]
    )
    if normalize_to_class:

    # Try to find ISs of the same lipid class
        class_candidates = [
            x for x in candidates
            if x["Class"] == feature["Class"]
        ]

        # If none exist, fall back to all candidates
        if not class_candidates:
            class_candidates = candidates

        # Choose the closest IS
        best_IS = min(
            [
                (IS_distance(feature, IS), IS)
                for IS in class_candidates
            ],
            key=lambda x: x[0]
        )
    
    if (
        normalize_to_group
        and sample_group is not None
        and len(scores) > 1
    ):

        best_score = scores[0][0]
        second_score = scores[1][0]

        if abs(best_score == second_score):


            return choose_lowest_cv_IS(
                sample_group,
                [
                    scores[0][1],
                    scores[1][1]
                ]
            )

    return scores[0][1]


def calculate_normalized_values(
    feature,
    selected_IS
):

    return {
        "normalized_precursor_intensity":
            feature["precursor_intensity"]
            / selected_IS.get("precursor_intensity"),

        "normalized_precursor_volume":
            feature["precursor_volume"]
            / selected_IS.get("precursor_volume")
    }


def normalize_to_IS(
    matches,
    rt_tolerance=30,
    mz_tolerance= 0.025,
    MS2_only=False,
    normalize_to_group=True,
    normalize_to_class=False
):

    normalized_features = []

    groups = defaultdict(list)
    IS_by_group = defaultdict(list)

    # Build groups once
    for feature in matches:

        group = extract_group(
            feature["file_id"]
        )

        feature["_group"] = group

        groups[group].append(feature)

        if feature.get("classification") == "internal_standard":

            if (
                not MS2_only
                or feature.get("has_ms2", False)
            ):
                IS_by_group[group].append(feature)


    # Normalize features
    for feature in matches:

        feature["normalization_flags"] = []

        group = feature["_group"]

        sample_group = [
            f
            for f in groups[group]
            if f.get("classification") != "internal_standard"
        ]

        candidates = IS_by_group[group]

        selected_IS = choose_best_IS(
            feature,
            candidates,
            rt_tolerance,
            mz_tolerance,
            sample_group=sample_group,
            normalize_to_group=normalize_to_group,
            normalize_to_class=normalize_to_class
        )

        if selected_IS is None:

            feature["normalized_precursor_intensity"] = None
            feature["normalized_precursor_volume"] = None

            feature["normalization_flag"] = (
                "No suitable IS found"
            )

            normalized_features.append(feature)
            continue


        if len(candidates) > 1:
            feature["normalization_flags"].append(
                "Multiple candidate IS found"
            )


        normalized = calculate_normalized_values(
            feature,
            selected_IS
        )

        feature.update(normalized)

        feature["normalized_to"] = (
            selected_IS["IS_match"]
        )

        normalized_features.append(feature)


    return normalized_features

In [ ]:
normalized_samples = normalize_to_IS(
    matches,
    rt_tolerance=30,
    mz_tolerance= 0.025,
    MS2_only=False,
    normalize_to_group=True,
    normalize_to_class=True
)

Calculate the dot product for the function that finds the best average MS2 value for MS1 peaks with multiple linked MS2 values and find the best average MS2 value for MS1 peaks with multiple linked MS2 values.

In [ ]:
# Function to calculate the dot product for the function that finds the best average MS2 value for MS1 peaks with multiple linked MS2 values
# This function should run AFTER link MSP
def dot_product_with_tolerance(mz1, int1, mz2, int2, tol=0.02):
    matched1, matched2 = [], []
    for i, m1 in enumerate(mz1):
        for j, m2 in enumerate(mz2):
            if abs(m1 - m2) <= tol:
                matched1.append(int1[i])
                matched2.append(int2[j])
                break
    if not matched1:
        return 0.0
    s1 = np.array(matched1); s2 = np.array(matched2)
    if np.linalg.norm(s1) == 0 or np.linalg.norm(s2) == 0:
        return 0.0
    s1 /= np.linalg.norm(s1); s2 /= np.linalg.norm(s2)
    return float(np.dot(s1, s2))

# Function to find the best average MS2 value for MS1 peaks with multiple linked MS2 values
def average_msms(df, top_n=3, mz_tolerance=0.02, mz_merge_thresh=0.01):
    required = ['peak_id','cluster_id','file_id','feature_id','msms_id',
                'ms2_rt','charge_state','ms2_sample_peaks','cent_mz','cent_intensity','precursor_mz',
                'precursor_rt','precursor_intensity','precursor_vol','total_intensity','total_volume',
                'average_ms1_mz','average_rt']

    missing = [c for c in required if c not in df.columns]

    assert not missing, f"Missing required columns: {missing}"

    results = []
    for (peak_id, file_id), group in df.groupby(['peak_id', 'file_id']):
        if len(group) < 2:
            row = group.iloc[0]
            results.append({
                'cluster_id': row['cluster_id'],
                'file_id' : file_id,
                'feature_id': row['feature_id'],
                'peak_id': peak_id,
                'avg_ms2_retention_time': row['ms2_rt'],
                'total_num_msms': 1,
                'charge_state': row['charge_state'],
                'avg_ms2_cent_peaks': list(zip(row['cent_mz'], row['cent_intensity'])),
                'avg_raw_ms2_sample_peaks': row['ms2_sample_peaks'],
                'avg_precursor_mz': row['precursor_mz'],
                'precursor_mz_list': row['precursor_mz'],
                'avg_precursor_rt': row['precursor_rt'],
                'precursor_rt_list': row['precursor_rt'],
                'avg_precursor_intensity' : row['precursor_intensity'],
                'precursor_intensity_list' : row['precursor_intensity'],
                'avg_precursor_vol': row['precursor_vol'],
                'precursor_vol_list': row['precursor_vol'],
                'dot_product_list': [],
                'avg_dot_product': 0.0,
                'entropy_similarity_list': [],
                'avg_entropy_similarity': 0.0,
                'ms2_sample_peaks' : row['ms2_sample_peaks']
            })
            continue

        msms_list = []
        for (_, r1), (_, r2) in combinations(group.iterrows(), 2):
            dot_product = dot_product_with_tolerance(r1['cent_mz'], r1['cent_intensity'],
                                            r2['cent_mz'], r2['cent_intensity'],
                                            tol=mz_tolerance)
            try:
                entropy_similarity = me.calculate_entropy_similarity(r1['ms2_sample_peaks'], r2['ms2_sample_peaks'])
            except Exception:
                entropy_similarity = None
            msms_list.append({
                'cluster_id': r1['cluster_id'],
                'feature_id': r1['feature_id'],
                'dot_product': dot_product,
                'entropy_similarity': entropy_similarity,
                'ms2_rt': r1['ms2_rt'],
                'cent_pairs': list(zip(r1['cent_mz'], r1['cent_intensity'])),
                'ms2_sample_peaks' : r1['ms2_sample_peaks'],
                'charge_state': r1['charge_state'],
                'precursor_mz': r1['precursor_mz'],
                'precursor_rt': r1['precursor_rt'],
                'precursor_intensity' : r1['precursor_intensity'],
                'precursor_vol': r1['precursor_vol'],
            })

        # filter out missing entropies
        msms_list = [m for m in msms_list if m['entropy_similarity'] is not None]
        if not msms_list:
            continue

        msms_list.sort(key=lambda x: x['entropy_similarity'], reverse=True)
        top_msms = msms_list[:top_n]

        all_cent_peaks = np.concatenate([np.array(m['cent_pairs']) for m in top_msms], axis=0)
        sorted_all_cent_peaks = all_cent_peaks[all_cent_peaks[:,0].argsort()]

        groups_current = []
        current = [sorted_all_cent_peaks[0]]
        for mz_i, intensity_i in sorted_all_cent_peaks[1:]:
            if abs(mz_i - current[-1][0]) <= mz_merge_thresh:
                current.append([mz_i, intensity_i])
            else:
                groups_current.append(np.array(current))
                current = [[mz_i, intensity_i]]
        groups_current.append(np.array(current))

        avg_cent_peaks = [[g[:,0].mean(), g[:,1].mean()] for g in groups_current]
        centroided_arr = np.array(avg_cent_peaks)
        centroided_arr_list = centroided_arr.tolist()

        all_raw_peaks = np.concatenate([np.array(m['ms2_sample_peaks']) for m in top_msms], axis=0)
        sorted_all_raw_peaks = all_raw_peaks[all_raw_peaks[:,0].argsort()]

        groups_current_raw = []
        current_raw = [sorted_all_raw_peaks[0]]
        for mz_i, intensity_i in sorted_all_cent_peaks[1:]:
            if abs(mz_i - current[-1][0]) <= mz_merge_thresh:
                current.append([mz_i, intensity_i])
            else:
                groups_current_raw.append(np.array(current_raw))
                current_raw = [[mz_i, intensity_i]]
        groups_current_raw.append(np.array(current_raw))

        avg_raw_peaks = [[g[:,0].mean(), g[:,1].mean()] for g in groups_current_raw]
        raw_arr = np.array(avg_raw_peaks)
        raw_arr_list = raw_arr.tolist()

        results.append({
            'cluster_id': [m['cluster_id'] for m in top_msms],
            'file_id' : file_id,
            'feature_id': [m['feature_id'] for m in top_msms],
            'peak_id': peak_id,
            'avg_ms2_retention_time': np.mean([m['ms2_rt'] for m in top_msms]),
            'total_num_msms': len(msms_list), # changed from (top_msms)
            'dot_product_list': [m['dot_product'] for m in top_msms],
            'avg_dot_product': np.mean([m['dot_product'] for m in top_msms]),
            'entropy_similarity_list': [m['entropy_similarity'] for m in top_msms],
            'avg_entropy_similarity': np.mean([m['entropy_similarity'] for m in top_msms]),
            'avg_ms2_cent_peaks': centroided_arr_list,
            'charge_state': [m['charge_state'] for m in top_msms],
            'precursor_mz_list': [m['precursor_mz'] for m in top_msms],
            'avg_precursor_mz' : np.mean([m['precursor_mz'] for m in top_msms]),
            'precursor_rt_list': [m['precursor_rt'] for m in top_msms],
            'avg_precursor_rt' : np.mean([m['precursor_rt'] for m in top_msms]),
            'precursor_intensity_list': [m['precursor_intensity'] for m in top_msms],
            'avg_precursor_intensity' : np.mean([m['precursor_intensity'] for m in top_msms]),
            'precursor_vol_list': [m['precursor_vol'] for m in top_msms],
            'avg_precursor_vol' : np.mean([m['precursor_vol'] for m in top_msms]),
            'avg_raw_ms2_sample_peaks' : raw_arr_list
        })

    return results

In [ ]:
results = average_msms(df, top_n=3, mz_tolerance=0.02, mz_merge_thresh=0.01)

In [ ]:
#avg_groups:
#Function to calculate the average normalized volume and intensity per group
# WARNING: This is pseudocode
def average_groups(normalized_samples):

    averaged_groups =[]

    groups = defaultdict(list)

    # Build groups once
    for feature in normalized_samples:

        group = extract_group(
            feature["file_id"]
        )

        feature["_group"] = group

        groups[group].append(feature)

        calculate CV between replicates in group

        if CV > CV_limit
            Flag : "Potentially anomalous" 
                keep CV
                    do not use feature in determining identified lipid or calculating average normalized intensity or volume

      calculate average normlized intensity and average normalized volume
            
       Take lipid identified in majority of samples as identified lipid
            If different lipids are identified in an equal number of replicates
                Flag : "Identified as multiple species" 
                    Take feature with highest score > min_score
                        If scores are equal or neither are above min_score
                            Flag: "Multiple low score identifications"   
                                Identified_lipid = "Unclear"
                        
    averaged_groups.append({
                    "group" : group,
                    "file_ids": file_ids,
                    "cluster_ids": cluster_ids,
                    "feature_ids": feature_id,
                    "peak_ids": peak_ids,
                    "avg_precursor_mz": avg_precursor_mz,
                    "avg_precursor_rt": avg_precursor_rt,
                    "charge_states": charge_states,
                    "avg_normalized_intensity": avg_normalized_intensity,
                    "avg_normalized_volume" : avg_normalized_intensity,
                    "identified_lipid" : identified_lipid, unknown, unclear
                    "CV" : CV,
                    "avg_CV" : avg_CV
                })
    return averaged_groups

#
#Function to calculate the CV between features in the same group
# This function already exists -> check if you can use pre-existing function
def calculate_cv

#Function to warn user of any potentially anomalous results
def warn()

In [ ]:
#avg_groups:
#Function to calculate the average normalized volume and intensity per group
def average_groups(normalized_samples, CV_limit):


    groups = defaultdict(list)

    # Build groups once
    for feature in normalized_samples:

        group = extract_group(feature["file_id"])

        feature["_group"] = group

        groups[group].append(feature)

        averaged_groups = []

        # Collect the normalized volumes and intensities for each group
        for group, features in groups.items():
            intensities = [
                f["normalized_intensity"]
                for f in features
                if f["normalized_intensity"] is not None
                ]

            volumes = [
                f["normalized_volume"]
                for f in features
                if f["normalized_volume"] is not None
            ]

        # Calculate CV between replicates in group
            mean_intensity = np.mean(intensities)

            intensity_cv = (
                    np.std(intensities, ddof=1)
                    / mean_intensity
                    * 100
                )

            mean_volume = np.mean(volumes)
            
            volume_cv = (
                    np.std(volumes, ddof=1)
                    / mean_volume
                     * 100
                )

            #Flag potentially anomalous features and remove them from averaging
            for feature in features:

                feature["flag"] = ""

                if feature["intensity_cv"] > CV_limit:
                    feature["flag"] = "Potentially anomalous"

                    valid_features = [
                        f
                        for f in features
                        if f["flag"] != "Potentially anomalous"
                    ]

                    avg_normalized_intensity = np.mean([
                            f["normalized_intensity"]
                            for f in valid_features
                            ])

                    avg_normalized_volume = np.mean([
                            f["normalized_volume"]
                            for f in valid_features
                            ])
                    avg_precursor_mz = np.mean([
                            f["precursor_mz"]
                            for f in valid_features
                            ])
                    
                    avg_precursor_rt = np.mean([
                            f["precursor_rt"]
                            for f in valid_features
                            ])

                    #Find identities
                    lipid_counts = defaultdict(list)

                    for feature in valid_features:

                        lipid = feature["identified_lipid"]

                        lipid_counts[lipid].append(feature)

                        max_count = max(
                            len(v)
                            for v in lipid_counts.values()
                        )

                        winners = [
                            lipid
                            for lipid, feats in lipid_counts.items()
                            if len(feats) == max_count
                        ]

                        identified_lipid = winners[0]

                        flag = "Identified as multiple species"

                        best_feature = None

                        for lipid in winners:

                            for feature in lipid_counts[lipid]:

                                if (
                                    best_feature is None
                                    or feature["score"] > best_feature["score"]
                                ):
                                    best_feature = feature

                                    if best_feature["score"] > CV_limit:
                                        identified_lipid = best_feature["identified_lipid"]

                                        identified_lipid = "Unclear"

                                        flag = "Multiple low score identifications"


                                        averaged_groups.append({

                                            "group": group,

                                            "file_ids": [
                                                f["file_id"]
                                                for f in features
                                            ],

                                            "cluster_ids": [
                                                f["cluster_id"]
                                                for f in features
                                            ],

                                            "feature_ids": [
                                                f["feature_id"]
                                                for f in features
                                            ],

                                            "peak_ids": [
                                                f["peak_id"]
                                                for f in features
                                            ],

                                            "avg_precursor_mz": avg_precursor_mz,

                                            "avg_precursor_rt": avg_precursor_rt,

                                            "charge_states": list({
                                                f["charge_state"]
                                                for f in features
                                            }),

                                            "avg_normalized_intensity":
                                                 avg_normalized_intensity,

                                            "avg_normalized_volume":
                                                avg_normalized_volume,

                                            "identified_lipid":
                                                identified_lipid,

                                            "intensity_cv" : intensity_cv,
                                            "volume_cv" : volume_cv,
                                            "flag": flag
                                        })

